# State-Space Models and Recurrence

This notebook builds the simplest linear state-space models that already expose the core SSM idea: a recurrence with a structured transition can be rewritten as a causal convolution with an impulse-response kernel.

## Motivation

State-space models offer a different route to long-range sequence modeling than quadratic attention. The S4 line of work makes that route practical by turning continuous-time linear dynamics into fast discrete sequence operators [@structured_state_spaces_2021]. A useful lens for this notebook is not “SSMs versus transformers” in the abstract, but the concrete computational question: when can we replace an explicit recurrent scan with a convolutional kernel without changing the output?

We stay inside the linear, input-independent setting in this task. That keeps the mathematics exact and separates the core recurrence mechanics from the selective or gated memory ideas deferred to the next notebook.

## Hypothesis

For scalar and diagonal linear recurrences with zero initial state, the recurrent output and the causal convolution output should match up to numerical precision when they share the same impulse response. Stability should depend on the discrete transition radius: modes strictly inside the unit circle should decay, while modes outside the unit circle should grow over time.

## Baseline

The baseline computation is the direct recurrent scan. At each timestep we update the latent state and emit the output immediately. This is the most transparent implementation because it mirrors the recurrence definition exactly.

The comparison system is the convolutional form obtained from the impulse response. If both implementations agree on the same deterministic inputs, then we have an executable check of the recurrence/convolution equivalence rather than only an algebraic claim.

## Metric

The main metric is the maximum absolute difference between recurrent outputs and convolution outputs:

$$
\max_t |y_t^{\text{recur}} - y_t^{\text{conv}}|.
$$

For the discretization step, we also compare the derived discrete parameters against the exact diagonal closed form. For stability, we track the final absolute state magnitude over a fixed horizon while sweeping transition values inside and outside the unit circle.

## Mathematical derivation

Start from a continuous-time linear state-space model

$$
\dot{x}(t) = A x(t) + B u(t), \qquad y(t) = C x(t),
$$

and assume the input is constant over each interval of width $\Delta$. Exact zero-order-hold discretization gives

$$
x_k = \bar{A} x_{k-1} + \bar{B} u_k,
$$

with

$$
\bar{A} = e^{\Delta A}, \qquad \bar{B} = \left(\int_0^{\Delta} e^{\tau A} \, d\tau\right) B.
$$

When $A$ is diagonal, everything becomes elementwise. For one scalar mode with discrete parameters $(a, b, c)$,

$$
x_k = a x_{k-1} + b u_k, \qquad y_k = c x_k.
$$

Unrolling the recurrence yields

$$
x_k = a^{k+1} x_{-1} + \sum_{j=0}^{k} a^{k-j} b u_j,
$$

so under zero initial state,

$$
y_k = \sum_{j=0}^{k} c b a^{k-j} u_j.
$$

That is exactly a causal convolution with impulse response

$$
h_\ell = c b a^{\ell}, \qquad \ell \ge 0.
$$

For a diagonal latent system with modes $a_i$, input couplings $b_i$, and output weights $c_i$, the scalar output kernel becomes

$$
h_\ell = \sum_i c_i b_i a_i^{\ell}.
$$

This decomposition is the notebook’s core bridge: diagonal linear recurrence and causal convolution are two implementations of the same operator.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.state_space import (
    diagonal_impulse_response,
    diagonal_linear_recurrence,
    diagonal_recurrence_convolution_outputs,
    discretize_diagonal_state_space,
    scalar_impulse_response,
    scalar_linear_recurrence,
    scalar_recurrence_convolution_outputs,
)

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## PyTorch implementation

The helper module implements four pieces of the linear SSM story:

1. scalar recurrence and its impulse response
2. diagonal recurrence and its impulse response
3. recurrent-versus-convolution comparison helpers
4. exact diagonal discretization from continuous to discrete time

We start with one scalar mode and then lift the same idea to a diagonal latent state.

In [ ]:
scalar_inputs = torch.tensor([1.0, -1.0, 0.5, 0.25], dtype=torch.float64)
scalar_transition = torch.tensor(0.7, dtype=torch.float64)
scalar_input_gain = torch.tensor(1.2, dtype=torch.float64)
scalar_output_gain = torch.tensor(0.8, dtype=torch.float64)

scalar_states, scalar_outputs = scalar_linear_recurrence(
    scalar_inputs,
    transition=scalar_transition,
    input_gain=scalar_input_gain,
    output_gain=scalar_output_gain,
)
scalar_impulse = scalar_impulse_response(
    transition=scalar_transition,
    input_gain=scalar_input_gain,
    output_gain=scalar_output_gain,
    length=scalar_inputs.numel(),
)
scalar_recurrent, scalar_convolution = scalar_recurrence_convolution_outputs(
    scalar_inputs,
    transition=scalar_transition,
    input_gain=scalar_input_gain,
    output_gain=scalar_output_gain,
)

diagonal_inputs = torch.tensor([1.0, 0.0, -2.0, 1.5, -0.5], dtype=torch.float64)
diagonal_transition = torch.tensor([0.95, 0.6, -0.3], dtype=torch.float64)
diagonal_input_gain = torch.tensor([1.0, -0.5, 0.75], dtype=torch.float64)
diagonal_output_gain = torch.tensor([0.4, 1.2, -0.8], dtype=torch.float64)

diagonal_states, diagonal_outputs = diagonal_linear_recurrence(
    diagonal_inputs,
    transition=diagonal_transition,
    input_gain=diagonal_input_gain,
    output_gain=diagonal_output_gain,
)
diagonal_impulse = diagonal_impulse_response(
    transition=diagonal_transition,
    input_gain=diagonal_input_gain,
    output_gain=diagonal_output_gain,
    length=diagonal_inputs.numel(),
)
diagonal_recurrent, diagonal_convolution = diagonal_recurrence_convolution_outputs(
    diagonal_inputs,
    transition=diagonal_transition,
    input_gain=diagonal_input_gain,
    output_gain=diagonal_output_gain,
)

print("scalar states:", scalar_states)
print("scalar impulse:", scalar_impulse)
print("scalar max error:", (scalar_recurrent - scalar_convolution).abs().max().item())
print()
print("diagonal first two states:")
print(diagonal_states[:2])
print("diagonal impulse:", diagonal_impulse)
print("diagonal max error:", (diagonal_recurrent - diagonal_convolution).abs().max().item())

## Numerical checks

The first check is exactness: recurrence and convolution should agree within floating-point tolerance for both the scalar and diagonal cases. The second check verifies the closed-form discretization. The third check sweeps transitions around the unit circle to show the stability boundary directly.

In [ ]:
torch.testing.assert_close(scalar_recurrent, scalar_convolution, atol=1e-12, rtol=1e-12)
torch.testing.assert_close(diagonal_recurrent, diagonal_convolution, atol=1e-12, rtol=1e-12)

continuous_transition = torch.tensor([0.0, -1.0, 0.3], dtype=torch.float64)
continuous_input = torch.tensor([2.0, 3.0, -4.0], dtype=torch.float64)
discrete_transition, discrete_input = discretize_diagonal_state_space(
    continuous_transition,
    continuous_input,
    step=torch.tensor(0.5, dtype=torch.float64),
)

expected_transition = torch.tensor([1.0, torch.exp(torch.tensor(-0.5, dtype=torch.float64)), torch.exp(torch.tensor(0.15, dtype=torch.float64))])
expected_input = torch.tensor([
    1.0,
    3.0 * (1.0 - torch.exp(torch.tensor(-0.5, dtype=torch.float64))),
    -4.0 * (torch.exp(torch.tensor(0.15, dtype=torch.float64)) - 1.0) / 0.3,
], dtype=torch.float64)

torch.testing.assert_close(discrete_transition, expected_transition, atol=1e-12, rtol=1e-12)
torch.testing.assert_close(discrete_input, expected_input, atol=1e-12, rtol=1e-12)

def free_response_magnitudes(transition_value: float, horizon: int = 24) -> torch.Tensor:
    zeros = torch.zeros(horizon, dtype=torch.float64)
    states, _ = scalar_linear_recurrence(
        zeros,
        transition=torch.tensor(transition_value, dtype=torch.float64),
        input_gain=torch.tensor(0.0, dtype=torch.float64),
        output_gain=torch.tensor(1.0, dtype=torch.float64),
        initial_state=torch.tensor(1.0, dtype=torch.float64),
    )
    return states.abs()

transition_sweep = torch.tensor([0.25, 0.75, 0.99, 1.01, 1.25, -1.10], dtype=torch.float64)
final_magnitudes = []
for value in transition_sweep.tolist():
    final_magnitudes.append(free_response_magnitudes(value)[-1].item())

for value, magnitude in zip(transition_sweep.tolist(), final_magnitudes):
    region = "inside" if abs(value) < 1.0 else "outside"
    print(f"a={value:>5.2f} |a|={abs(value):>4.2f} ({region}) -> final |state|={magnitude:>10.6f}")

assert final_magnitudes[0] < 1e-6
assert final_magnitudes[1] < 1.0
assert final_magnitudes[2] < 1.0
assert final_magnitudes[3] > 1.0
assert final_magnitudes[4] > 1.0
assert final_magnitudes[5] > 1.0

print()
print("All numerical checks passed.")

## Ablations

A useful ablation is to move the transition spectrum while holding the input and output couplings fixed. Modes near the unit circle produce much longer impulse tails than heavily damped modes. That matters because long memory is beneficial only if the system remains numerically controlled.

In [ ]:
ablation_systems = {
    "short_memory": torch.tensor([0.2, 0.4, 0.6], dtype=torch.float64),
    "long_memory": torch.tensor([0.85, 0.92, 0.98], dtype=torch.float64),
    "mixed_with_unstable_mode": torch.tensor([0.85, 0.98, 1.05], dtype=torch.float64),
}
coupling_in = torch.tensor([1.0, 0.8, -0.6], dtype=torch.float64)
coupling_out = torch.tensor([0.5, 1.0, -0.75], dtype=torch.float64)

for name, transition in ablation_systems.items():
    impulse = diagonal_impulse_response(
        transition=transition,
        input_gain=coupling_in,
        output_gain=coupling_out,
        length=8,
    )
    print(name)
    print("  first 5 impulse coefficients:", impulse[:5])
    print("  l1 tail over 8 steps:", impulse.abs().sum().item())

## Assumptions

- The input is piecewise constant over each discretization interval.
- The latent transition is scalar or diagonal, so the mode interactions are independent.
- All equivalence checks use zero initial state unless noted otherwise.
- We study linear, input-independent dynamics only; selective or gated state updates are intentionally excluded from this notebook.

## Risks

- Exact equivalence here can create false confidence: full S4 implementations need careful parameterization and efficient kernel construction beyond this toy diagonal setting.
- Near-unit-circle modes preserve memory but can become numerically delicate over long horizons.
- A diagonal transition is pedagogically clean but cannot express cross-channel mixing on its own.

## Failure criteria

- The notebook fails if recurrent and convolution outputs disagree beyond floating-point tolerance.
- It fails if the discretized parameters do not match the exact diagonal closed form.
- It fails if the stability sweep does not separate decaying modes with $|a| < 1$ from growing modes with $|a| > 1$.
- It fails if any section drifts into selective or input-dependent recurrence, which belongs to the next notebook rather than this one.

## Exercises

1. Starting from $x_k = a x_{k-1} + b u_k$, derive the impulse response for output gain $c$.
2. Why does a diagonal latent transition make the convolution kernel a sum of mode-wise exponentials?
3. For $a = 0.98$ and $a = 1.02$, which system keeps a longer memory over 100 steps, and which one is stable?
4. Show that when the continuous-time transition is zero, the discretized input gain reduces to $\Delta B$.
5. Give one benefit and one limitation of studying diagonal SSMs before selective recurrence.

Companion exercise files: `exercises/research/20_state_space_models.md` and `exercises/research/solutions/20_state_space_models_solutions.md`.

## References

- Gu et al., *Efficiently Modeling Long Sequences with Structured State Spaces* [@structured_state_spaces_2021]
- Peng et al., *RWKV: Reinventing RNNs for the Transformer Era* [@rwkv_2023]